# polyline conversion
This section is all about turning the poly lines file into a array of json/dict elements that we can feed into the later steps

In [6]:
import json

In [7]:
import networkx as nx

In [8]:
import itertools

In [9]:
from pathlib import Path

In [10]:
# this file is newline delimited where each line is a sequence of points that are in a branch of the tree
polylines = Path("skel-poly.polylines.txt").read_text().split("\n")

In [11]:
# we need to iterate over the lines of the file which is a branch, and from that collect the values into groups of 3
# these are the coordinates x,y,z for a single point
vert_sets = []
for line in polylines:
    parts = line.strip().split(" ")[1:]
    vs=[]
    for i in range(0,len(parts),3):
        [x,y,z] = parts[i:i+3]
        vs.append({"x":float(x),"y":float(y),"z":float(z)})
    vert_sets.append(vs)
vert_sets

[[{'x': 0.337464, 'y': 0.0545325, 'z': 5.66947},
  {'x': 0.343796, 'y': 0.0555746, 'z': 5.67673},
  {'x': 0.359046, 'y': 0.0581846, 'z': 5.69304},
  {'x': 0.370663, 'y': 0.0602877, 'z': 5.70452},
  {'x': 0.390882, 'y': 0.0641331, 'z': 5.72338},
  {'x': 0.406726, 'y': 0.0674922, 'z': 5.73555}],
 [{'x': 0.629061, 'y': -1.25667, 'z': 6.00466},
  {'x': 0.62462, 'y': -1.25441, 'z': 6.01175},
  {'x': 0.614614, 'y': -1.24871, 'z': 6.02893},
  {'x': 0.60785, 'y': -1.24492, 'z': 6.04552},
  {'x': 0.599498, 'y': -1.24035, 'z': 6.06537},
  {'x': 0.596805, 'y': -1.23841, 'z': 6.07219},
  {'x': 0.586833, 'y': -1.22796, 'z': 6.09835},
  {'x': 0.583422, 'y': -1.22387, 'z': 6.1087},
  {'x': 0.579264, 'y': -1.20889, 'z': 6.13936},
  {'x': 0.575726, 'y': -1.19473, 'z': 6.1696},
  {'x': 0.577726, 'y': -1.17326, 'z': 6.20899}],
 [{'x': 0.249965, 'y': 0.0717785, 'z': 5.43966},
  {'x': 0.267876, 'y': 0.072204, 'z': 5.42557},
  {'x': 0.279455, 'y': 0.0739865, 'z': 5.40969},
  {'x': 0.287594, 'y': 0.0754676, 

In [12]:
# write this out in case we want to start from here later, or use on the web
Path("lines.json").write_text(json.dumps(vert_sets))

188459

In [13]:
lines = json.loads(Path("lines.json").read_text())

In [14]:
lines

[[{'x': 0.337464, 'y': 0.0545325, 'z': 5.66947},
  {'x': 0.343796, 'y': 0.0555746, 'z': 5.67673},
  {'x': 0.359046, 'y': 0.0581846, 'z': 5.69304},
  {'x': 0.370663, 'y': 0.0602877, 'z': 5.70452},
  {'x': 0.390882, 'y': 0.0641331, 'z': 5.72338},
  {'x': 0.406726, 'y': 0.0674922, 'z': 5.73555}],
 [{'x': 0.629061, 'y': -1.25667, 'z': 6.00466},
  {'x': 0.62462, 'y': -1.25441, 'z': 6.01175},
  {'x': 0.614614, 'y': -1.24871, 'z': 6.02893},
  {'x': 0.60785, 'y': -1.24492, 'z': 6.04552},
  {'x': 0.599498, 'y': -1.24035, 'z': 6.06537},
  {'x': 0.596805, 'y': -1.23841, 'z': 6.07219},
  {'x': 0.586833, 'y': -1.22796, 'z': 6.09835},
  {'x': 0.583422, 'y': -1.22387, 'z': 6.1087},
  {'x': 0.579264, 'y': -1.20889, 'z': 6.13936},
  {'x': 0.575726, 'y': -1.19473, 'z': 6.1696},
  {'x': 0.577726, 'y': -1.17326, 'z': 6.20899}],
 [{'x': 0.249965, 'y': 0.0717785, 'z': 5.43966},
  {'x': 0.267876, 'y': 0.072204, 'z': 5.42557},
  {'x': 0.279455, 'y': 0.0739865, 'z': 5.40969},
  {'x': 0.287594, 'y': 0.0754676, 

In [15]:
# checking if we can use this as a key in a dictionary
f"{lines[0][0]}"

"{'x': 0.337464, 'y': 0.0545325, 'z': 5.66947}"

In [16]:
# helpful package for things related to iteration, will use it for pairwise iteration later
import itertools

# Building a networkx graph
This lets us build paths out of the points between roots and leafs

In [17]:

# make an empty networkx graph
g = nx.Graph()

In [18]:
len(g.nodes)

0

In [19]:
# the cell underneath makes a map of the vertices so that we don't include any repeat ones when we make our graph
# n is the new id for the node in the graph

In [20]:
n=1
mp = {}
for i,l in enumerate(lines):
    for point in l:
        key = f"{point}"
        if mp.get(key,-1) !=-1:
            continue
        mp[key] =n
        n+=1

In [21]:
len(mp.keys())

3719

In [22]:
mp

{"{'x': 0.337464, 'y': 0.0545325, 'z': 5.66947}": 1,
 "{'x': 0.343796, 'y': 0.0555746, 'z': 5.67673}": 2,
 "{'x': 0.359046, 'y': 0.0581846, 'z': 5.69304}": 3,
 "{'x': 0.370663, 'y': 0.0602877, 'z': 5.70452}": 4,
 "{'x': 0.390882, 'y': 0.0641331, 'z': 5.72338}": 5,
 "{'x': 0.406726, 'y': 0.0674922, 'z': 5.73555}": 6,
 "{'x': 0.629061, 'y': -1.25667, 'z': 6.00466}": 7,
 "{'x': 0.62462, 'y': -1.25441, 'z': 6.01175}": 8,
 "{'x': 0.614614, 'y': -1.24871, 'z': 6.02893}": 9,
 "{'x': 0.60785, 'y': -1.24492, 'z': 6.04552}": 10,
 "{'x': 0.599498, 'y': -1.24035, 'z': 6.06537}": 11,
 "{'x': 0.596805, 'y': -1.23841, 'z': 6.07219}": 12,
 "{'x': 0.586833, 'y': -1.22796, 'z': 6.09835}": 13,
 "{'x': 0.583422, 'y': -1.22387, 'z': 6.1087}": 14,
 "{'x': 0.579264, 'y': -1.20889, 'z': 6.13936}": 15,
 "{'x': 0.575726, 'y': -1.19473, 'z': 6.1696}": 16,
 "{'x': 0.577726, 'y': -1.17326, 'z': 6.20899}": 17,
 "{'x': 0.249965, 'y': 0.0717785, 'z': 5.43966}": 18,
 "{'x': 0.267876, 'y': 0.072204, 'z': 5.42557}": 19,

In [23]:
# here we use pairwise iteration as we go over the lines so that we can add the nodes to the networks and make edges between them
for l in lines:
    for (start,end) in itertools.pairwise(l):
        s_id =mp[f"{start}"] # recall this resolves to the n value for the vertex 
        e_id = mp[f"{end}"]
        g.add_nodes_from([(s_id,start),(e_id,end)])
        g.add_edge(s_id,e_id)
        

In [24]:
list(g.nodes)

[1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 88,
 89,
 90,
 91,
 92,
 93,
 94,
 95,
 96,
 97,
 98,
 99,
 100,
 101,
 102,
 103,
 104,
 105,
 106,
 107,
 108,
 109,
 110,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 125,
 126,
 127,
 128,
 129,
 130,
 131,
 132,
 133,
 134,
 135,
 136,
 137,
 138,
 139,
 140,
 141,
 142,
 143,
 144,
 145,
 146,
 147,
 148,
 149,
 150,
 151,
 152,
 153,
 154,
 155,
 156,
 157,
 158,
 159,
 160,
 161,
 162,
 163,
 164,
 165,
 166,
 167,
 168,
 169,
 170,
 171,
 172,
 173,
 174,
 175,
 176,
 177,
 178,
 179,
 180,
 181,
 182,
 183,
 184,
 185

In [25]:
# we need to look for all the points that only have one connection to another, these will be the tips of the tree (roots, and leaves)
yvals = []
for n in g:
    if g.degree(n) ==1:
        attrs = nx.get_node_attributes(g,"y")
        yvals.append([n,attrs[n]]) # add the node identifier n and the y coordinate to a yvals list

In [26]:
#this will return only the y value when we sort the list
def sorted(x):
    return x[1]

In [27]:
# sort based on the y value element for each
yvals.sort(key=sorted)

In [28]:
yvals

[[7, -1.25667],
 [2716, -1.2273],
 [1292, -1.21856],
 [2717, -1.21823],
 [2694, -0.683607],
 [1240, -0.618103],
 [198, -0.605787],
 [3366, -0.604102],
 [2798, -0.559495],
 [740, -0.512219],
 [2754, -0.487526],
 [486, -0.427162],
 [752, -0.330391],
 [780, -0.302314],
 [1005, -0.298805],
 [3116, -0.279901],
 [788, -0.268554],
 [3086, -0.252322],
 [2264, -0.227905],
 [3035, -0.17527],
 [2368, -0.165235],
 [851, -0.145825],
 [862, -0.138699],
 [2171, -0.137049],
 [2025, -0.123742],
 [2625, -0.0473967],
 [951, -0.0425888],
 [1887, -0.0112755],
 [811, -0.00859495],
 [2379, 0.00109],
 [1933, 0.0107039],
 [2108, 0.0279671],
 [3244, 0.0299058],
 [791, 0.0402568],
 [764, 0.0482931],
 [795, 0.0534555],
 [1, 0.0545325],
 [3062, 0.0560632],
 [2024, 0.0614826],
 [702, 0.067089],
 [1476, 0.109868],
 [935, 0.118199],
 [993, 0.126642],
 [2837, 0.145387],
 [37, 0.156059],
 [971, 0.159488],
 [1615, 0.166558],
 [1589, 0.176504],
 [1274, 0.177723],
 [2017, 0.177878],
 [2114, 0.178155],
 [1705, 0.181768],
 

In [29]:
# get the nodes below -1 y, they are the roots of the tree
# NOTE when working with different 3d models this logic may need to change
# generate shortest paths from them
# add all of them to a list
roots = [e[0] for e in yvals[:4]]
roots

[7, 2716, 1292, 2717]

In [30]:
# the rest of the ids are the tips of the leaves
tips = [e[0] for e in yvals[4:]]
tips

[2694,
 1240,
 198,
 3366,
 2798,
 740,
 2754,
 486,
 752,
 780,
 1005,
 3116,
 788,
 3086,
 2264,
 3035,
 2368,
 851,
 862,
 2171,
 2025,
 2625,
 951,
 1887,
 811,
 2379,
 1933,
 2108,
 3244,
 791,
 764,
 795,
 1,
 3062,
 2024,
 702,
 1476,
 935,
 993,
 2837,
 37,
 971,
 1615,
 1589,
 1274,
 2017,
 2114,
 1705,
 1702,
 1197,
 2239,
 62,
 2656,
 2712,
 2164,
 1184,
 2078,
 2179,
 3423,
 1961,
 3415,
 2074,
 2937,
 73,
 3362,
 3437,
 3413,
 2635,
 1091,
 620,
 2993,
 3288,
 2567,
 340,
 730,
 418,
 2484,
 977,
 796,
 3397,
 1783,
 731,
 362,
 485,
 2755,
 1659,
 289,
 1506,
 367,
 940,
 355,
 1571,
 1925,
 732,
 2682,
 1923,
 1513,
 1844,
 1697,
 3434,
 266,
 1946,
 253,
 3280,
 1955,
 994,
 278,
 825,
 2063,
 2911,
 2515,
 84,
 627,
 3341,
 3137,
 3308,
 1086,
 2051,
 3122,
 1572,
 2336,
 2985,
 205,
 326,
 3289,
 3191,
 351,
 2369,
 1181,
 1874,
 2325,
 1122,
 3154,
 1563,
 616,
 3388,
 3468,
 2623,
 396,
 2454,
 1809,
 1706,
 2432,
 2721,
 1300,
 1803]

In [31]:
len(tips)

146

In [32]:
import tqdm

In [33]:
# Here's the most important part from this section
# we are using the shortest_path method to calculate the paths between roots and leaves
paths = []
pairings = [[r,t] for r in roots for t in tips]
for r,t in tqdm.tqdm(pairings):
    simple_paths = nx.shortest_path(g,r,t)
    # add to a structure that we can write out later, and use for .obj generation
    paths.append({"source":r,"target":t,"paths":list(itertools.pairwise(simple_paths))})
    # break # if we want to test we can uncomment this and only run one iteration

100%|███████████████████████████████████████| 584/584 [00:00<00:00, 1071.01it/s]


In [34]:
Path("test_path_export.json").write_text(json.dumps(paths))

903332

In [35]:
paths[0]

{'source': 7,
 'target': 2694,
 'paths': [(7, 8),
  (8, 9),
  (9, 10),
  (10, 11),
  (11, 12),
  (12, 13),
  (13, 14),
  (14, 15),
  (15, 16),
  (16, 17),
  (17, 1299),
  (1299, 2655),
  (2655, 2654),
  (2654, 2653),
  (2653, 2652),
  (2652, 2651),
  (2651, 2650),
  (2650, 2649),
  (2649, 1385),
  (1385, 2648),
  (2648, 2607),
  (2607, 2606),
  (2606, 2605),
  (2605, 2604),
  (2604, 2588),
  (2588, 2608),
  (2608, 2609),
  (2609, 2610),
  (2610, 2611),
  (2611, 2612),
  (2612, 2613),
  (2613, 2614),
  (2614, 2615),
  (2615, 2616),
  (2616, 2617),
  (2617, 2618),
  (2618, 2619),
  (2619, 2620),
  (2620, 2621),
  (2621, 2622),
  (2622, 2694)]}

In [36]:




# grab the attribute lists
x_attr = nx.get_node_attributes(g,"x")
y_attr = nx.get_node_attributes(g,"y")
z_attr = nx.get_node_attributes(g,"z")

# make a collection for the specific paths
vert_paths=[]
for p in paths:
    text = ""
    # this helps us only have one reference to the 3d coordinates per node id
    vert_row_map ={}
    # this is the node index we will track this so we can write out the vertices in the right order later
    n=0
    # iterate over the paths for each starting root
    # note these are the pairs of coordinates show above, they are the start and end point of a edge in the graph
    for nodes in p["paths"]:
        
        # for each node in the pair 
        for node_id in nodes:
            key = node_id
            # if this node id doesnt have coords in the map add them
            # is line returns -1 it means that node isn't in the map yet
            # if it is in the map we run into the continue part of the conditional
            if vert_row_map.get(key,-1) !=-1:
                continue
            x = x_attr[node_id]
            y = y_attr[node_id]
            z = z_attr[node_id]
            # make sure we can access the index of the node in the path in the value as well so we can write it out in order as .obj
            vert_row_map[key] =[n,x,y,z]
            # add one to the node index counter
            n+=1
    # add the path to the vertex paths
    vert_paths.append(vert_row_map)

In [37]:
# make a obj writer for each curve in the file that connects a root with a leaf

# can use i to map between the two lists vert_paths, and paths since they have the same order
for i,p in enumerate(paths):
    lines = ""
    verts = ""

    # lets write the vertex rows of the file first
    # get the collection of x,y,z points for this path and the order they should be written to the file
    vmap = vert_paths[i]
    for k,v in vmap.items():
        # note we use 1: to skip the first value in the list because that is the n value and we want to only have x,y,z in the row separated by ' '
        verts +=f"\nv {' '.join([str(e) for e in v[1:]])}"

    # now we will write the lines
    for start_id,end_id in p["paths"]:
        # use the vert_path_ map to get the right edge line values
        start_id_mapped = vmap[start_id][0]
        end_id_mapped = vmap[end_id][0]
        lines +=f"\nl {start_id_mapped} {end_id_mapped}"
    Path(f"test_verts_{i}.obj").write_text(verts+lines)



In [38]:
!rm test*.obj

## making simulation files

Here we will begin by outputting json files that can be visualized on the web

Then we will work on writing data out as animating files that work in blender

In [39]:
# start by making sort of a test file
import numpy as np

In [40]:
import json
from pathlib import Path 

In [41]:
# this makes a random data array where we have 100 time steps
# each time step is like a table of 500 rows with 3 columns x,y,z
timearr = np.random.random((100,500,3))

# write this out to json
Path("test.json").write_text(json.dumps([[{"x":e[0],"y":e[1],"z":e[2]} for e in arr ] for arr in timearr ]))

3890621

In [42]:
# make particles start at the beginning of one of the paths, and then with each time step they move to a new part on the path

x_attr = nx.get_node_attributes(g,"x")
y_attr = nx.get_node_attributes(g,"y")
z_attr = nx.get_node_attributes(g,"z")
# this can hold all the paths that the points go after we iterate over them in time

# the only way I could think to do this was first make tracks or a list of the x,y,z points that a particle is as it goes along a path


tracks=[]
# iterate over the paths
for path in paths:
    # get the pairs in a path
    point_pairs = path["paths"]

    # this will be the route a single point has to take, 
    # we can think of it as the steps in 3d space a particle traveling the path will take over time
    position_times = []

    # again start and end are node indices in the graph, just integer values
    for start,end in point_pairs:

        # we only need the start because the end will be a start in the next interation
        x = x_attr[start]
        y = y_attr[start]
        z = z_attr[start]
        # add the data as a dictionary element
        # this is easy to work with and convert to tables or arrays later
        position_times.append(dict(x=x,y=y,z=z))
    tracks.append(position_times)    

In [43]:
# this is how many branches there are in the tree
len(tracks)

584

In [44]:
# this is the length of the first branch in steps for a particle
len(tracks[0])

41

In [45]:
# the structure right now is
# tracks, then time
# we need to change to time, then tracks, this matches better our random data written out above as test.json

# we need to find the max length of all the tracks
max([len(e) for e in tracks])


158

In [46]:
# now we will iterate and collect the positions of each point at each time 
time_steps =[]
# i is sort of the time step (also the index into the position collections held in each track)
for i in range(158):
    # make somewhere we cna store the positions for this tiem
    positions =[]
    # iterate over the tracks
    for track in tracks:
        # make sure our current time isn't greater than the track has entries for
        if i < len(track):
            # if we have an entry, write it out
            positions.append(track[i])
    # add this collection of all the positions on all the branches to a time_step
    time_steps.append(positions)

In [47]:

# write this out for visualization prototyping
Path("test.json").write_text(json.dumps([[{"x":e["x"],"y":e["y"],"z":e["z"]} for e in arr ] for arr in time_steps ]))

2992262

In [48]:
import random

In [49]:
random.choice(range(20))

8

In [50]:
tracks[0]

[{'x': 0.629061, 'y': -1.25667, 'z': 6.00466},
 {'x': 0.62462, 'y': -1.25441, 'z': 6.01175},
 {'x': 0.614614, 'y': -1.24871, 'z': 6.02893},
 {'x': 0.60785, 'y': -1.24492, 'z': 6.04552},
 {'x': 0.599498, 'y': -1.24035, 'z': 6.06537},
 {'x': 0.596805, 'y': -1.23841, 'z': 6.07219},
 {'x': 0.586833, 'y': -1.22796, 'z': 6.09835},
 {'x': 0.583422, 'y': -1.22387, 'z': 6.1087},
 {'x': 0.579264, 'y': -1.20889, 'z': 6.13936},
 {'x': 0.575726, 'y': -1.19473, 'z': 6.1696},
 {'x': 0.577726, 'y': -1.17326, 'z': 6.20899},
 {'x': 0.557317, 'y': -1.14876, 'z': 6.23403},
 {'x': 0.56315, 'y': -1.12174, 'z': 6.23572},
 {'x': 0.561245, 'y': -1.10506, 'z': 6.2382},
 {'x': 0.559677, 'y': -1.07972, 'z': 6.23892},
 {'x': 0.558871, 'y': -1.05718, 'z': 6.23855},
 {'x': 0.558168, 'y': -1.03689, 'z': 6.23808},
 {'x': 0.557797, 'y': -1.01687, 'z': 6.23697},
 {'x': 0.557188, 'y': -0.979862, 'z': 6.23515},
 {'x': 0.557138, 'y': -0.955427, 'z': 6.23065},
 {'x': 0.557102, 'y': -0.930365, 'z': 6.21286},
 {'x': 0.560999,


## making the USD files
Here we convert our working simulation data to a format we can bring into blender automatically


In [64]:
# Akshat TODO, read in the velocity data as pd dataframe
# pick a single column from the depths,
# normalize it
import pandas as pd
df = pd.read_csv("sap_flow.csv")

df["date"] =pd.to_datetime(df['Datetime'], format='%m/%d/%Y %H:%M')

mid_speeds = df["Vc_15mm_cm/hr"]
mid_speeds

mid_min = mid_speeds.min()
mid_max = mid_speeds.max()



normalized = (mid_speeds - mid_min)/(mid_max-mid_min)

normalized
# note, look for another way to use the pandas series but with reindexing , unfortunately reset_index turns it into a whole dataframe again
stepped = np.array((normalized*9 + 1).dropna().astype("int16"))

stepped

array([5, 5, 5, ..., 5, 5, 5], shape=(14265,), dtype=int16)

In [ ]:
# so we need to increase the number of time steps to be as long as the csv data is
# in terms of veocity we might need to find a way to relate the scale of the model with number of steps along a path
# worry about that later

# need a way to think about the code from a "particle" perspective
# it would have a path it's on
# it would have a velocity (how many steps to move on the path)  
# 

# make the particle pick a path/track to start with using random choice 
timesteps= []
particles = []
def create_particle(tracks):
    track = random.choice(tracks)
    # NOTE this is probably something we want to base on the min and max values of the size of the tree, too much noise totally distorts the tree
    noise = np.random.random((3))/50
    return dict(
        track = track,
        # keep understandign of length of track
        track_length = len(track),
        starting_position = track[0],
        current_position = track[0],
        offset=noise,
        # note that we can track next position and then interpolate also
        index = 0
    )

# make a "move particles" function
# this function also potentially removes elements from the p_list
# will return a new list
def move_particles(p_list,series,simulation_step_index):
    new_list =[] 
    for particle in p_list:
        index = particle["index"]
        track =particle["track"]
        track_length = particle["track_length"]
        # if we can continue, particle hasn't fallen off the track
        # Akshat TODO, change the parts of the code that depend index +1 to be index+ some velocity mapped step [1,10]
        
        step_size = series[simulation_step_index]
        if index+step_size < track_length-1:
            # this is where we will think about stepping more than one index place through the track when velocity is higher
            # use the sca
            # Akshat TODO, change the parts of the code that depend index +1 to be index+ some velocity mapped step [1,10]
            
            next_index = index +step_size
            particle["current_position"] = track[next_index]
            particle["index"] = next_index
            # if we wanted to interpolate here's where we would get the next and then use some sort of parametric form
            new_list.append(particle)
    return new_list

    
def add_particles(p_list,number_to_add,tracks):
    new_particles = [create_particle(tracks) for i in range(number_to_add)]
    p_list.extend(new_particles)
    return p_list


# make a "take snapshot" function that gets all positions and writes them out to the 
def take_snapshot(tsteps,p_list):
    # injecting a little bit of noise
    tsteps.append([
        {
            "x": p["current_position"]["x"] + p["offset"][0], # could add more forces if we wanted to make more dynamic,
            "y":p["current_position"]["y"] + p["offset"][1],
            "z":p["current_position"]["z"] + p["offset"][2]
            
        }
        for p in p_list
    ])

simulation_steps = stepped.shape[0]
number_particles_to_start = 500
number_to_add_per_step=10
particles = add_particles(particles,number_particles_to_start,tracks)
for time_index in tqdm.tqdm(range(simulation_steps)):
    # move the particles
    moved_particles = move_particles(particles,stepped,time_index)
    particles = moved_particles
    # add new ones
    added_particles = add_particles(particles,number_to_add_per_step,tracks)
    particles = added_particles
    # snapshot
    take_snapshot(timesteps,particles)
    
# at the end make the output match the way the usdc write out pattern works
# have an array of time steps
# each time step has collection of point positions


len(timesteps)

len(timesteps[1])


import pandas as pd


# see if we can delete the previous stage, maeks it easier to work with cell iteratively
try:
    print(stage)
    del stage
except:
    pass

# this cell does a lot of work
from pxr import Usd,UsdGeom,Vt,Sdf
from pathlib import Path
import numpy as np
import argparse
# name our output file

In [68]:
name="first_sap_data_render"
# create the variable we add points to, it's called a stage in usd
stage = Usd.Stage.CreateNew(f"{name}.usdc")
pts = UsdGeom.Points.Define(stage,"/mypoints")

import numpy as np

# get attributes we can write to
points = pts.GetPointsAttr()
widths = pts.GetWidthsAttr()

# establish the length of the timeseries
stage.SetStartTimeCode(0)
stage.SetEndTimeCode(len(timesteps))

# loop over our timesteps
for i,json_points in enumerate(timesteps):
    # if we end up attaching data to each poitn we will use these names
    # names = arr.dtype.names[3:]
    # pvars={}
    # arr[:,1], arr[:,2]


    # convert from list of dicts to a table, then to an array
    df = pd.DataFrame(json_points)
    arr = df.to_numpy()



    # write the array data to the attribute
    # print(arr)
    points.Set(time=i,value=Vt.Vec3fArray.FromNumpy(np.array([arr[:,0],arr[:,1],arr[:,2]]).T))


    # again we will use this later when we attach data to the points

    # for name in names:
    #   # skip the "_" prefaced names that stand for offset balancing in pcd binary
    #     if "skip" in name:
    #       continue
        # pvar = UsdGeom.PrimvarsAPI(pts).CreatePrimvar(name,Sdf.ValueTypeNames.FloatArray,"vertex")
        # pvar.Set(time=i,value= Vt.FloatArray(arr[name].astype("float64")))
        # pvars[name] = pvar
    

    

stage.Save()

In [66]:
len(timesteps)

14265

In [67]:
timesteps[0:5]

[[{'x': np.float64(0.5785699459483743),
   'y': np.float64(-1.1643665342362837),
   'z': np.float64(6.210010287284361)},
  {'x': np.float64(0.6325400672751179),
   'y': np.float64(-1.1759409831666536),
   'z': np.float64(6.23699876759814)},
  {'x': np.float64(0.5994540851457104),
   'y': np.float64(-1.218639400205301),
   'z': np.float64(6.076752900735725)},
  {'x': np.float64(0.5893851324145964),
   'y': np.float64(-1.1633020180909288),
   'z': np.float64(6.2125514165228415)},
  {'x': np.float64(0.6345101135832226),
   'y': np.float64(-1.173031834820861),
   'z': np.float64(6.249251229854096)},
  {'x': np.float64(0.5883723853785825),
   'y': np.float64(-1.1567252342089147),
   'z': np.float64(6.213059458406025)},
  {'x': np.float64(0.6115583990361632),
   'y': np.float64(-1.2196467890637988),
   'z': np.float64(6.088308809089942)},
  {'x': np.float64(0.5859582693024082),
   'y': np.float64(-1.153417988950332),
   'z': np.float64(6.223794813811899)},
  {'x': np.float64(0.64076466302504

In [61]:
stepped[1]

KeyError: 1